# KanoFarm AI: train the Plant Doctor (starter model)
**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run each cell in order.
This trains on PlantDoc (CC BY 4.0). It covers **tomato, potato and pepper** well enough to test the pipeline. It does **not** cover cowpea, groundnut, rice, sorghum, millet, cassava or onion, and it is **not validated for Nigerian farms**.

In [ ]:
!nvidia-smi | head -5

In [ ]:
# EDIT THIS: your GitHub repo URL
REPO = "https://github.com/YOUR-NAME/kanofarm-ai"
!git clone $REPO
%cd kanofarm-ai

In [ ]:
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset

In [ ]:
# Keep only the crops we can support. Add ,maize ONLY after you add your own maize___healthy images.
!python -m ml.preprocessing.prepare_plantdoc --src PlantDoc-Dataset --dst data/plantdoc --crops tomato,potato,pepper

**Read the output above.** `unmatched_folders` lists folders the script could not map. If your desired classes are missing, folder names differ and the map file needs adjusting.

In [ ]:
!python -m ml.training.build_splits --src data/plantdoc --dst data/split --presplit

In [ ]:
!python -m ml.training.train --data data/split --out ml/models/v1 \
  --dataset-name "PlantDoc (CC BY 4.0)" --dataset-version "github-2020" --epochs 15

In [ ]:
import json
card = json.load(open("ml/models/v1/model_card.json"))
m = card["metrics"]
print("test images:", m["n"], "| accuracy:", m["accuracy"], "| macro F1:", m["macro_f1"])
print("\nPer-class (worst recall first):")
for k, v in sorted(m["per_class"].items(), key=lambda kv: kv[1]["recall"]):
    print(f'{k:40s} recall={v["recall"]:.2f} precision={v["precision"]:.2f} n={v["support"]}')

**Look at the worst classes.** Low recall or tiny test counts mean the model is unreliable for them. Do not trust accuracy alone.

In [ ]:
!cp data/plantdoc/label_meta.json ml/models/v1/label_meta.json
!mkdir -p out/model && cp ml/models/v1/model.pt ml/models/v1/model_card.json ml/models/v1/label_meta.json out/model/
!cp ml/serving/hf_space/* out/ 2>/dev/null; ls -R out | head -20
!cd out && zip -r ../plant_doctor_v1.zip . >/dev/null
from google.colab import files
files.download("plant_doctor_v1.zip")